<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='14.%20file_uploads_and_static_assets.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#8592; Chapter 14: File Uploads</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#128218; Table of Contents</a>
  <a href='16.%20testing_flask_applications.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 16: Testing &#8594;</a>
</div>

# Chapter 15: Error Handling and Logging -- Graceful Failures

> *"Every app crashes sometimes. The question is: does it show a cryptic 500 page, or a friendly error page? And do YOU know about the crash? Error handling + logging is the difference between a professional app and an amateur one."*

## 🎯 The Big Picture

When Flask encounters a problem two things must happen simultaneously:

1. **The user** sees a friendly, on-brand error page -- no technical details exposed.
2. **You (the developer)** immediately know about the error with enough context to fix it.

Without proper error handling, users see raw Python tracebacks (a security risk). Without logging you fly blind -- you won't know the app is failing until a user emails you.

```text
User hits broken route
         |
         v
   Flask error handler  -->  renders 404.html  -->  User sees friendly page
         |
         v
   app.logger.error()   -->  log file / Sentry  -->  Developer gets alerted
```

This chapter covers both pillars: **custom error pages** and **structured logging**.

## 🧠 Core Concepts: The "Why"

### The Two Audiences for Errors

Think of a hospital emergency room:
- **Patients** hear: *"We're taking care of you."* -- reassuring, no jargon.
- **Doctors** get: *"SpO2 at 85%, administer O2 immediately"* -- detailed, actionable.

| Audience | What they see | What they need |
|----------|---------------|----------------|
| **Users** | Friendly HTML error page | Reassurance + what to do next |
| **Developers** | Full traceback in logs | Enough detail to reproduce and fix |

### The Golden Rule

> **Never show a Python traceback to a user in production. Never.**

A traceback reveals your file system structure, library versions (useful for attackers), and potentially sensitive config values.
### HTTP Error Codes

| Code | Meaning | When it happens |
|------|---------|-----------------|
| `400` | Bad Request | Malformed input |
| `401` | Unauthorized | Not logged in |
| `403` | Forbidden | Logged in but not allowed |
| `404` | Not Found | Wrong URL |
| `429` | Too Many Requests | Rate limited |
| `500` | Internal Server Error | Your code crashed |

## ⚡ Syntax & First Usage

### Registering Custom Error Handlers

The `@app.errorhandler()` decorator intercepts errors **before Flask sends its default response**. When a 404 occurs anywhere in the application (wrong URL, `abort(404)`, `.get_or_404()`), Flask looks for a registered handler and calls it.

**How it works:**
1. Flask receives a request
2. A 404 (or other) error is raised somewhere in the request pipeline
3. Flask checks: "Is there a registered `@errorhandler(404)`?"
4. If yes: calls the handler function, which returns a `Response`
5. If no: Flask returns its default HTML error page

**For web apps** — return a styled HTML template:
```python
@app.errorhandler(404)
def not_found(error):
    return render_template('errors/404.html'), 404
```

**For APIs** — return JSON (never HTML):
```python
@app.errorhandler(404)
def not_found(error):
    return jsonify(error=str(error)), 404
```

**The `error` parameter** is the exception object. For HTTP errors, it's a `werkzeug.exceptions.HTTPException` with:
- `error.code` — the HTTP status code (404)
- `error.name` — the human-readable name ("Not Found")
- `error.description` — the default message ("The requested URL was not found...")

> ❓ **What is a decorator?** A decorator is a function that wraps another function to add behaviour before or after it runs. `@app.errorhandler(404)` registers your handler function with Flask's error-handling system — it's called automatically when a 404 occurs, not when you call it directly.

In [ ]:

from flask import Flask, render_template_string

app = Flask(__name__)

@app.errorhandler(404)
def not_found_error(error):
    html = "<h1>404 - Page Not Found</h1><p>The page you requested doesn't exist.</p>"
    return render_template_string(html), 404   # status code is the second return value

@app.errorhandler(500)
def internal_error(error):
    # db.session.rollback()  # critical in a real app -- resets broken DB state
    html = "<h1>500 - Something Went Wrong</h1><p>We are looking into it!</p>"
    return render_template_string(html), 500

print("Error handlers registered!")
print()
print("Key points:")
print("  1. The handler receives the error object as its argument")
print("  2. The HTTP status code MUST be the second return value")
print("  3. Without handlers, Flask returns default plain-text error pages")


### Using Real Templates (Blueprint Pattern)

In real applications, error handlers belong in a dedicated **Blueprint** rather than in the main application file. This keeps the codebase organised and makes error handling modular and reusable.

**Why a separate Blueprint for errors?**
- The main `create_app()` factory stays clean — it just registers blueprints
- Error templates live in their own folder (`errors/templates/errors/`)
- You can test error handlers in isolation
- Consistent with how Flask applications are structured at scale

**File structure:**
```text
app/
├── errors/
│   ├── __init__.py       ← Blueprint definition
│   ├── handlers.py       ← @errorhandler functions
│   └── templates/
│       └── errors/
│           ├── 404.html  ← "Page Not Found" template
│           ├── 403.html  ← "Access Forbidden" template
│           └── 500.html  ← "Server Error" template
```

**`app/errors/__init__.py`:**
```python
from flask import Blueprint

errors_bp = Blueprint('errors', __name__, template_folder='templates')

from . import handlers  # import after blueprint to avoid circular imports
```

**`app/errors/handlers.py`:**
```python
from flask import render_template
from . import errors_bp
from .. import db

@errors_bp.app_errorhandler(404)
def not_found(error):
    return render_template('errors/404.html'), 404

@errors_bp.app_errorhandler(403)
def forbidden(error):
    return render_template('errors/403.html'), 403

@errors_bp.app_errorhandler(500)
def server_error(error):
    db.session.rollback()   # CRITICAL: prevent broken DB transactions
    return render_template('errors/500.html'), 500
```

**Note the difference:** Use `@app.errorhandler()` on the app object, or `@blueprint.app_errorhandler()` (note the `app_` prefix) on a blueprint. Using `@blueprint.errorhandler()` (without `app_`) only catches errors **within that blueprint's routes**.

Register in `create_app()`:
```python
from .errors import errors_bp
app.register_blueprint(errors_bp)
```

In [ ]:

# app/errors/__init__.py  -- Blueprint-based error handling (best practice)

code_blueprint = """
from flask import Blueprint, render_template
from app import db

errors = Blueprint("errors", __name__)

@errors.app_errorhandler(404)
def not_found_error(error):
    return render_template("errors/404.html"), 404

@errors.app_errorhandler(403)
def forbidden_error(error):
    return render_template("errors/403.html"), 403

@errors.app_errorhandler(500)
def internal_error(error):
    db.session.rollback()          # ALWAYS reset broken DB state
    return render_template("errors/500.html"), 500
"""

# templates/errors/404.html
template_404 = """
{% extends "base.html" %}
{% block title %}Page Not Found{% endblock %}
{% block content %}
  <div class="error-container text-center py-5">
    <h1 class="display-1 text-muted">404</h1>
    <h2>Oops - that page does not exist</h2>
    <p class="text-muted">The page you requested could not be found.</p>
    <a href="{{ url_for('main.index') }}" class="btn btn-primary mt-3">Go Home</a>
  </div>
{% endblock %}
"""

print("Blueprint error handler pattern:")
print(code_blueprint)
print("templates/errors/404.html:")
print(template_404)


## 🔬 Deep Dive

### Custom HTTP Exceptions

Flask's error hierarchy starts with `werkzeug.exceptions.HTTPException` — all HTTP error classes (`NotFound`, `Forbidden`, etc.) inherit from it.

**The error hierarchy:**
```text
BaseException
└── Exception
    └── werkzeug.exceptions.HTTPException
        ├── BadRequest (400)
        ├── Unauthorized (401)
        ├── Forbidden (403)
        ├── NotFound (404)
        ├── Conflict (409)
        ├── UnprocessableEntity (422)
        └── InternalServerError (500)
```

**Custom exception classes** inherit from `HTTPException` to add domain-specific meaning. Instead of `abort(404)` everywhere, you raise a named exception that documents intent:

```python
# Instead of: abort(404)  # what kind of 404?
# Use:        raise PostNotFound()  # explicit, searchable, meaningful

from werkzeug.exceptions import HTTPException

class PostNotFound(HTTPException):
    code = 404
    description = "The requested post does not exist or has been deleted."

class DuplicateEmail(HTTPException):
    code = 409
    description = "An account with this email address already exists."

class PostLocked(HTTPException):
    code = 403
    description = "This post is locked and cannot be modified."
```

Register a handler for each (or let the generic `HTTPException` handler catch them all):
```python
@app.errorhandler(PostNotFound)
def handle_post_not_found(error):
    return render_template('errors/404.html', message=error.description), 404
```

In [ ]:

from werkzeug.exceptions import HTTPException

class PostNotFound(HTTPException):
    code = 404
    description = "The blog post does not exist or has been deleted."

class InsufficientPermissions(HTTPException):
    code = 403
    description = "You do not have permission to perform this action."

class RateLimitExceeded(HTTPException):
    code = 429
    description = "Too many requests. Please try again in a minute."

example_route = """
@app.route("/post/<int:post_id>")
def view_post(post_id):
    post = Post.query.get(post_id)
    if post is None:
        raise PostNotFound()                    # descriptive, self-documenting
    if not post.is_published and post.author != current_user:
        raise InsufficientPermissions()
    return render_template("post.html", post=post)
"""

print("Custom exceptions:")
for exc in [PostNotFound, InsufficientPermissions, RateLimitExceeded]:
    print(f"  {exc.__name__:<30} HTTP {exc.code}")
print()
print("Usage in routes:")
print(example_route)
print("Benefits: self-documenting, single place to update, testable with try/except.")


### `abort()` vs Raising Exceptions vs `.get_or_404()`

Flask provides multiple ways to stop request processing and return an error. Understanding the differences helps you choose the most readable approach:

| Method | How it works | Best for |
|---|---|---|
| `abort(404)` | Immediately raises an `HTTPException` | Simple inline errors |
| `raise PostNotFound()` | Raises your custom exception | Semantic, named errors |
| `Post.query.get_or_404(id)` | SQLAlchemy shortcut | Database lookups |
| `.first_or_404()` | SQLAlchemy query shortcut | Filtered queries |

**`abort()` — immediate stop:**
```python
from flask import abort

@app.route('/posts/<int:id>')
def post(id):
    if id < 1:
        abort(400)  # Bad Request — stops execution here
    post = Post.query.get(id)
    if not post:
        abort(404)  # Not Found
    return render_template('post.html', post=post)
```

**`.get_or_404()` — SQLAlchemy shortcut:**
```python
# These two are equivalent:
post = Post.query.get(id)
if not post:
    abort(404)

# Cleaner:
post = Post.query.get_or_404(id, description=f"Post {id} not found")
```

**`.first_or_404()` — for filtered queries:**
```python
post = Post.query.filter_by(slug=slug, published=True).first_or_404(
    description=f"No published post with slug '{slug}'"
)
```

**When to use each:**
- `abort()` → quick, inline, when the error is a simple HTTP code
- `raise CustomException()` → when you want the error to be self-documenting and searchable in code
- `.get_or_404()` / `.first_or_404()` → always, for database lookups (cleaner than the if/abort pattern)

In [ ]:

# Three ways to trigger error responses

abort_example = """
# abort() -- best for simple inline one-liners
from flask import abort

@app.route("/admin/")
def admin_dashboard():
    if not current_user.is_admin:
        abort(403)
    return render_template("admin/dashboard.html")
"""

raise_example = """
# raise CustomException -- best for semantic clarity
@app.route("/post/<int:id>/delete", methods=["POST"])
def delete_post(id):
    post = Post.query.get_or_404(id)
    if post.author_id != current_user.id:
        raise InsufficientPermissions()    # clear, self-documenting
    db.session.delete(post)
    db.session.commit()
    return redirect(url_for("main.index"))
"""

shortcut_example = """
# get_or_404() -- SQLAlchemy convenience shortcuts
post = Post.query.get_or_404(post_id)
user = User.query.filter_by(username=name).first_or_404()
# Automatically abort(404) if not found; return object if found.
"""

print("abort()      -- immediate, one-liner for simple cases")
print("raise        -- named exceptions for semantic clarity")
print("get_or_404() -- best shortcut when fetching by primary key")
print()
for name, code in [("abort()", abort_example), ("raise", raise_example), ("shortcut", shortcut_example)]:
    print(f"=== {name} ===")
    print(code)


### Flask's Built-in Logger

Flask exposes Python's standard library `logging` module as `app.logger` — a pre-configured `Logger` instance with your app's name.

**Python's logging architecture:**

```text
app.logger  (Logger)
    │
    ├──► StreamHandler  → stdout/stderr
    ├──► FileHandler    → log file
    └──► SentryHandler  → Sentry cloud
         │
         ▼
      Formatter  (defines the log line format)
```

**The 5 log levels** — use the right one for the right event:

| Level | When to use | Example |
|---|---|---|
| `DEBUG` | Detailed tracing for development | `"Querying DB with filter: {...}"` |
| `INFO` | Normal operation events | `"User 42 logged in"`, `"Post 99 created"` |
| `WARNING` | Something unexpected but recoverable | `"Disk 90% full"`, `"3 failed login attempts"` |
| `ERROR` | Serious failure, request failed | `"Payment processor timeout"`, `"DB query failed"` |
| `CRITICAL` | System-level failure requiring immediate action | `"Database unreachable"`, `"Out of memory"` |

**Basic usage in routes:**
```python
from flask import current_app

@app.route('/login', methods=['POST'])
def login():
    user = authenticate(email, password)
    if not user:
        current_app.logger.warning(f"Failed login for email: {email} from IP: {request.remote_addr}")
        return render_template('login.html', error="Invalid credentials"), 401
    
    current_app.logger.info(f"User {user.id} ({user.email}) logged in")
    login_user(user)
    return redirect(url_for('main.index'))
```

> ⚠️ **Never log sensitive data**: No passwords, no tokens, no credit card numbers. If you log `email` for failed logins, that's fine. If you log the password they tried, you've created a security disaster.

In [ ]:

log_levels = {
    "DEBUG":    10,   # Granular debugging -- suppressed in production
    "INFO":     20,   # Normal operational events
    "WARNING":  30,   # Unexpected but handled
    "ERROR":    40,   # Serious problem, some functionality lost
    "CRITICAL": 50,   # System-wide failure
}

print("Log level hierarchy (lowest to highest severity):")
print("=" * 50)
for level, value in log_levels.items():
    bar = "█" * (value // 5)
    print(f"  {level:<10} ({value:2d})  {bar}")

usage = """
# Inside routes (use current_app.logger in blueprints)
app.logger.debug("Detailed debug -- shown only in development")
app.logger.info("User 42 logged in successfully")
app.logger.warning("Disk usage at 90% -- check storage")
app.logger.error("Payment timeout for order 99", exc_info=True)
app.logger.critical("Database pool exhausted")
"""
print()
print("Usage:", usage)
print("Setting level to INFO suppresses all DEBUG messages.")
print("Setting level to WARNING suppresses INFO and DEBUG.")


### Setting Up File Logging with `RotatingFileHandler`

**Why file logging?** `print()` and `stdout` logging disappear when the process restarts. File logs persist, can be searched, and can be analysed for patterns.

**`RotatingFileHandler` vs `StreamHandler`:**

| Handler | Where logs go | Use when |
|---|---|---|
| `StreamHandler` | stdout/stderr | Development, Docker (container logs captured by orchestrator) |
| `RotatingFileHandler` | File on disk | Traditional servers, VMs, when you need persistent logs |
| `TimedRotatingFileHandler` | File, rotated by time | When you prefer daily/weekly log files |

**Why `RotatingFileHandler`?** Logs grow without bound. A production app can generate gigabytes of logs. `RotatingFileHandler` automatically renames the current log file when it hits `maxBytes` and starts a new one, keeping at most `backupCount` old files:

```text
app.log        ← current, being written to
app.log.1      ← previous rotation
app.log.2      ← older
...
app.log.10     ← oldest (then deleted)
```

**Production-ready logging setup:**
```python
import logging
from logging.handlers import RotatingFileHandler
import os

def configure_logging(app):
    if not app.debug:
        # Ensure logs directory exists
        os.makedirs('logs', exist_ok=True)
        
        # File handler: 10MB per file, keep 10 backups = max 110MB
        file_handler = RotatingFileHandler(
            'logs/app.log',
            maxBytes=10 * 1024 * 1024,  # 10 MB
            backupCount=10
        )
        
        # Formatter: timestamp + level + logger name + message
        formatter = logging.Formatter(
            '%(asctime)s %(levelname)-8s %(name)s: %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        )
        file_handler.setFormatter(formatter)
        file_handler.setLevel(logging.INFO)
        
        app.logger.addHandler(file_handler)
        app.logger.setLevel(logging.INFO)
        app.logger.info('Application startup')
```

Call `configure_logging(app)` inside `create_app()` after the app is configured.

In [ ]:

import logging
from logging.handlers import RotatingFileHandler

logging_setup = """
def configure_logging(app):
    if not app.debug and not app.testing:
        import os
        if not os.path.exists("logs"):
            os.mkdir("logs")

        handler = RotatingFileHandler(
            "logs/app.log",
            maxBytes=10 * 1024 * 1024,  # 10 MB per file
            backupCount=10              # app.log + app.log.1 ... .10 = 110 MB max
        )
        formatter = logging.Formatter(
            "%(asctime)s %(levelname)s: %(message)s [in %(pathname)s:%(lineno)d]"
        )
        handler.setFormatter(formatter)
        handler.setLevel(logging.INFO)

        app.logger.addHandler(handler)
        app.logger.setLevel(logging.INFO)
        app.logger.info("Application startup")
"""

print("Production logging configuration:")
print(logging_setup)
print("Example log output:")
print("  2024-01-15 14:23:01 INFO: Application startup [in app/__init__.py:45]")
print("  2024-01-15 14:23:05 INFO: User 42 logged in [in app/auth/routes.py:87]")
print("  2024-01-15 14:25:10 WARNING: Failed login for test@ex.com [in routes.py:102]")
print("  2024-01-15 14:30:22 ERROR: 500 Internal Error [in app/errors/handlers.py:18]")


### Logging Throughout Your Application

**Structured logging** means attaching consistent context (user ID, request ID, endpoint) to every log message so you can trace an issue across multiple entries.

**What to log at each level** in a Flask web app:

```python
# INFO — log all significant business events
app.logger.info(f"User {user.id} created post {post.id}")
app.logger.info(f"User {user.id} logged in from {request.remote_addr}")
app.logger.info(f"Password reset email sent to {email}")

# WARNING — log anomalies that need monitoring
app.logger.warning(f"Failed login attempt #{attempts} for {email} from {request.remote_addr}")
app.logger.warning(f"Rate limit exceeded: {request.remote_addr}")
app.logger.warning(f"Disk usage at {disk_usage}% — approaching limit")

# ERROR — log failures that need investigation
app.logger.error(f"Payment failed for user {user.id}: {str(e)}", exc_info=True)
app.logger.error(f"Email delivery failed: {email}", exc_info=True)

# exc_info=True — attaches the full traceback to the log entry
```

**JSON structured logging** (recommended for production):

Plain text logs are hard to search and analyse. JSON logs can be ingested by log aggregators (ELK Stack, Datadog, Splunk, CloudWatch) and queried like a database:

```python
import json, logging

class JSONFormatter(logging.Formatter):
    def format(self, record):
        return json.dumps({
            "time":    self.formatTime(record),
            "level":   record.levelname,
            "message": record.getMessage(),
            "logger":  record.name,
            "module":  record.module,
            "line":    record.lineno,
        })

# Usage: grep '{"level": "ERROR"' app.log | jq '.message'
```

> 💡 **Request ID pattern**: Generate a unique ID per request and attach it to every log message in that request. This lets you trace all log entries for a single failing request across multiple log lines.

In [ ]:

route_logging = """
from flask import current_app

# Authentication -- always log these events
@auth.route("/login", methods=["POST"])
def login():
    user = User.query.filter_by(email=form.email.data).first()
    if user and user.check_password(form.password.data):
        login_user(user)
        current_app.logger.info(f"User {user.id} logged in")
        return redirect(url_for("main.index"))

    current_app.logger.warning(
        f"Failed login: {form.email.data} from {request.remote_addr}"
    )
    return render_template("auth/login.html")


# Destructive operations -- log with user ID and resource ID
@posts.route("/post/<int:id>/delete", methods=["POST"])
@login_required
def delete_post(id):
    post = Post.query.get_or_404(id)
    current_app.logger.info(f"Post {post.id} deleted by user {current_user.id}")
    db.session.delete(post)
    db.session.commit()
    return redirect(url_for("main.index"))


# Error handler -- log with full traceback
@errors.app_errorhandler(500)
def internal_error(error):
    db.session.rollback()
    current_app.logger.error(f"500 error: {error}", exc_info=True)
    return render_template("errors/500.html"), 500
"""

print("What to log:")
print("  ALWAYS: Authentication events (logins, failures, logouts)")
print("  ALWAYS: Destructive actions (deletes, bulk updates)")
print("  ALWAYS: External service calls (email, payment, API)")
print("  ALWAYS: All 500 errors with exc_info=True for full traceback")
print()
print("NEVER log: passwords, API keys, tokens, credit card numbers, SSNs")
print()
print(route_logging)


### Debug Mode vs Production Mode

Flask's debug mode (`DEBUG=True`) enables features that make development fast but create severe security risks in production.

| Feature | DEBUG=True (development) | DEBUG=False (production) |
|---|---|---|
| **Interactive debugger** | ✅ Browser-based Werkzeug debugger | ❌ Disabled |
| **Auto-reload** | ✅ Restarts on code changes | ❌ Disabled |
| **Error tracebacks** | ✅ Shown in browser | ❌ Hidden (500 page shown instead) |
| **PIN protection** | Debugger requires PIN | N/A |
| **Performance** | Slower | Faster |
| **Security** | ⚠️ Dangerous — see below | ✅ Safe |

**Why `DEBUG=True` in production is catastrophic:**

The Werkzeug interactive debugger allows executing **arbitrary Python code** in the browser through a debug console. If an attacker triggers an unhandled exception, they can:
1. Open the debugger console
2. Execute `import os; os.system('rm -rf /')` (or worse)
3. Read environment variables containing secrets
4. Exfiltrate your database

This is essentially **Remote Code Execution (RCE)** — the most severe class of vulnerability.

**Environment-based configuration:**

```python
# config.py
class DevelopmentConfig:
    DEBUG = True
    TESTING = False

class ProductionConfig:
    DEBUG = False      # NEVER True in production
    TESTING = False

# Set via environment variable
config = {
    'development': DevelopmentConfig,
    'production': ProductionConfig,
}
app.config.from_object(config[os.environ.get('FLASK_ENV', 'production')])
```

> 🔒 **Rule**: Never set `DEBUG=True` in production. Use environment variables to configure your app, not hardcoded values in source code.

In [ ]:

comparison = """
+----------------------+-------------------------------+------------------------------+
| Feature              | DEBUG=True (dev only!)        | DEBUG=False (production)     |
+----------------------+-------------------------------+------------------------------+
| Exception display    | Werkzeug interactive debugger | Custom error handler runs    |
| Auto-reload          | Yes, on code changes          | No                           |
| Error traceback      | Shown in browser              | Logged only, never shown     |
| Remote code exec     | Possible via debugger         | N/A                          |
| Performance          | Slower                        | Faster                       |
| Security             | NEVER use in production       | Safe                         |
+----------------------+-------------------------------+------------------------------+
"""

config_code = """
class ProductionConfig:
    DEBUG   = False   # CRITICAL: never True in production
    TESTING = False
    SECRET_KEY = os.environ["SECRET_KEY"]
    SQLALCHEMY_DATABASE_URI = os.environ["DATABASE_URL"]

class DevelopmentConfig:
    DEBUG   = True    # interactive debugger, auto-reload

class TestingConfig:
    DEBUG   = False
    TESTING = True    # disables error catching for cleaner test assertions
    SQLALCHEMY_DATABASE_URI = "sqlite:///:memory:"
    WTF_CSRF_ENABLED = False
"""

print(comparison)
print("Why DEBUG=True in production is a CRITICAL vulnerability:")
print("  Werkzeug debugger allows REMOTE CODE EXECUTION via browser")
print("  Exposes full source code, file paths, config values")
print("  Attackers actively scan for Flask debug mode on the internet")
print()
print("Config pattern:")
print(config_code)


### ⚖️ Comparison: `print()` vs `app.logger`

Beginners reach for `print()` naturally — it's instant and familiar. But `app.logger` is strictly better for any real application:

| Feature | `print()` | `app.logger` |
|---|---|---|
| **Output destination** | stdout only | File, stdout, Sentry, etc. |
| **Log levels** | None (all equal) | DEBUG, INFO, WARNING, ERROR, CRITICAL |
| **Timestamps** | None | Automatic |
| **Can be silenced** | No (always prints) | Yes (set minimum level) |
| **Searchable** | Only with grep | Yes, especially with JSON format |
| **Production-safe** | No (pollutes stdout) | Yes |
| **Traceback support** | Manual `import traceback` | `exc_info=True` |

**The `print()` problem in production:**
```python
# These print statements will appear in your production server logs mixed with
# everything else, with no timestamps, levels, or context:
print("User logged in")          # OK during dev
print(f"Error: {e}")             # Where? When? Which request?
print(user.password)             # 💀 Password in logs forever
```

**The right way:**
```python
# Context-rich, levelled, filterable, structured
app.logger.info(f"User {user.id} logged in from {request.remote_addr}")
app.logger.error(f"Payment failed for order {order.id}", exc_info=True)
```

**When is `print()` acceptable?** In one-off scripts, management commands, and data migrations — not in request handlers or application code.

In [ ]:

comparison = """
+----------------------+---------------------------+------------------------------+
| Feature              | print()                   | app.logger                   |
+----------------------+---------------------------+------------------------------+
| Output destination   | stdout only               | Console + files + Sentry +…  |
| Severity levels      | None (all identical)      | DEBUG/INFO/WARNING/ERROR     |
| Suppress by level    | Must delete manually      | setLevel() in config         |
| Timestamps           | No                        | Yes, via Formatter           |
| Module + line number | No                        | Yes, via Formatter           |
| Production-safe      | Clutters stdout           | Fully configurable           |
| Integration          | None                      | Sentry, ELK, CloudWatch, …   |
+----------------------+---------------------------+------------------------------+
"""

bad = """
# BAD -- print() in production
@app.route("/login", methods=["POST"])
def login():
    print(f"DEBUG: login for {request.form['email']}")   # no timestamp
    print(f"DEBUG: password: {request.form['password']}") # SECURITY NIGHTMARE
"""

good = """
# GOOD -- structured logging
@app.route("/login", methods=["POST"])
def login():
    current_app.logger.debug("Login form submitted")          # filtered in prod
    if not user:
        current_app.logger.warning(f"Failed: {email[:3]}***") # partial only
"""

print(comparison)
print("BAD (print):", bad)
print("GOOD (logger):", good)


### Sentry: Production Error Monitoring

**What is Sentry?** Sentry is a cloud service that automatically captures **unhandled exceptions** from your application, enriches them with context (user, request, breadcrumbs), and sends you alerts in real time.

**What logging alone can't do:**
- Logs are passive — you read them when you suspect a problem
- Sentry is active — it alerts you the moment a new error occurs
- Sentry groups duplicate errors so you see "this error occurred 47 times" instead of 47 separate log lines
- Sentry captures the full request context, user info, and environment variables automatically

**The production error monitoring flow:**
```text
User hits unhandled exception
         │
         ▼
Sentry SDK captures: exception + traceback
                   + request (URL, method, headers, body)
                   + user (email, ID — if you set it)
                   + breadcrumbs (recent log messages)
         │
         ▼
Sentry groups with similar errors
         │
         ▼
Alert sent (email/Slack/PagerDuty) → Developer investigates
```

**Setup:**
```python
# pip install sentry-sdk[flask]

import sentry_sdk
from sentry_sdk.integrations.flask import FlaskIntegration

def create_app():
    app = Flask(__name__)
    
    # Initialize Sentry BEFORE registering routes/blueprints
    if not app.debug:
        sentry_sdk.init(
            dsn=os.environ['SENTRY_DSN'],   # from sentry.io project settings
            integrations=[FlaskIntegration()],
            traces_sample_rate=0.1,          # 10% of requests for performance monitoring
            send_default_pii=False,          # don't send PII by default
        )
    
    # ... rest of create_app()
    return app
```

**Attaching user context** (so you know which user saw the error):
```python
import sentry_sdk
from flask_login import current_user

@app.before_request
def set_sentry_user():
    if current_user.is_authenticated:
        sentry_sdk.set_user({"id": current_user.id, "email": current_user.email})
```

> 💡 Sentry has a generous free tier (5,000 errors/month). Add it to every production Flask app — you'll thank yourself the first time a user reports an error you didn't know about.

In [ ]:

sentry_setup = """
# 1. Install: pip install sentry-sdk[flask]

# 2. Initialize in create_app() BEFORE registering routes:
import sentry_sdk
from sentry_sdk.integrations.flask import FlaskIntegration

def create_app(config_name="default"):
    app = Flask(__name__)

    if not app.debug and not app.testing:
        sentry_sdk.init(
            dsn=os.environ.get("SENTRY_DSN"),       # never hardcode!
            integrations=[FlaskIntegration()],
            traces_sample_rate=0.1,                 # 10% of requests for perf
            environment="production",
            release=os.environ.get("GIT_SHA")       # link errors to deploys
        )

    return app
"""

features = [
    "Full Python tracebacks with local variable values at each stack frame",
    "Request context (URL, method, headers, POST data)",
    "User context (configure with sentry_sdk.set_user())",
    "Release tracking -- know which deploy introduced a bug",
    "Email/Slack/PagerDuty alerts for new issues",
    "Smart grouping -- same error counted, not duplicated",
    "Performance tracing -- slow routes and N+1 queries",
]

print("Sentry setup:")
print(sentry_setup)
print("What Sentry captures automatically:")
for f in features:
    print(f"  OK: {f}")
print()
print("Free tier: ~5,000 errors/month. Perfect for most small-to-medium projects.")


## 🧪 What If?

### What if `db.session.rollback()` is missing in the 500 handler?

**Understanding the problem**: SQLAlchemy maintains a database **transaction** across a request. If an error occurs mid-transaction (e.g., a database write fails), the transaction is left in a "broken" state — it can't be used for further queries until it's explicitly rolled back.

**Without `rollback()` in the 500 handler:**

```text
Request 1: POST /create-post
  → Start DB transaction
  → INSERT into posts... fails (e.g., unique constraint violation)
  → Exception raised → 500 handler called
  → No rollback() → transaction still "open" and broken

Request 2: GET /posts  (same DB connection from pool!)
  → Reuses the broken connection from Request 1
  → SQLAlchemy: "this session has an unfinished transaction"
  → Raises "InvalidRequestError: Can't reconnect until invalid transaction is rolled back"
  → Another 500!

Request 3, 4, 5... all fail the same way
```

**With `rollback()` in the 500 handler:**

```python
@app.errorhandler(500)
def server_error(error):
    db.session.rollback()   # Clean up the broken transaction
    app.logger.error(f"500 error: {error}", exc_info=True)
    return render_template('errors/500.html'), 500
```

The rollback releases the broken transaction, allowing the connection to be returned to the pool clean and usable for future requests.

> 📌 **Always include `db.session.rollback()` in your 500 handler.** Without it, a single database error can cascade into every subsequent request failing.

In [ ]:

explanation = """
WITHOUT rollback() in the 500 handler:
---------------------------------------
Request 1: POST /create-post
  -> db.session.add(new_post)       <-- transaction started
  -> notify_followers()             <-- raises SMTPException!
  -> 500 handler fires              <-- session NOT cleaned up
  -> DB session left in invalid state

Request 2 (same Gunicorn worker): GET /my-posts
  -> Post.query.all()
     sqlalchemy.exc.InvalidRequestError:
     "Can't reconnect until invalid transaction is rolled back"
  -> User sees ANOTHER 500! Cascading failure.

WITH rollback() in the 500 handler:
--------------------------------------
Request 1: POST /create-post
  -> db.session.add(new_post)
  -> notify_followers()             <-- raises SMTPException!
  -> 500 handler: db.session.rollback()  <-- session cleaned up
  -> Friendly error page shown

Request 2: Works perfectly.
"""

handler = """
@errors.app_errorhandler(500)
def internal_error(error):
    db.session.rollback()    # non-negotiable
    current_app.logger.error(f"500 error: {error}", exc_info=True)
    return render_template("errors/500.html"), 500
"""

print(explanation)
print("Correct 500 handler:")
print(handler)


### What if you log sensitive data? / What if logs grow forever?

**Sensitive data in logs — a career-ending mistake:**

Logs are often stored less securely than databases. They're shared with ops teams, shipped to third-party log aggregators, and backed up to object storage. A single `app.logger.info(f"Password: {password}")` puts every user's password in a potentially public log file forever.

```python
# CATASTROPHICALLY DANGEROUS — these log to disk forever
app.logger.info(f"Login attempt: email={email}, password={password}")  # 💀
app.logger.debug(f"User token: {token}")                               # 💀
app.logger.info(f"Card number: {card}")                               # 💀 (also illegal)

# SAFE — log only identifiers, never secrets
app.logger.info(f"Login attempt for email: {email}")       # ✅
app.logger.warning(f"Failed login for user_id: {user.id}") # ✅
app.logger.info(f"Token issued for user: {user.id}")       # ✅
```

**Rules for what to log:**
- ✅ User IDs, request IDs, timestamps, status codes, durations
- ✅ Email addresses (for auth events — acceptable risk)
- ❌ Passwords, password hashes, tokens, API keys
- ❌ Full credit card numbers, CVV codes (violates PCI-DSS)
- ❌ Social security / national ID numbers (violates GDPR/HIPAA)

**Log files growing forever — disk exhaustion:**

Without `RotatingFileHandler`, a busy production app can fill a disk in days:

```text
Without rotation:
Week 1:  app.log  100 MB
Week 4:  app.log  400 MB
Week 12: app.log  1.2 GB → disk full → server crashes → data loss
```

**Solution: Always use `RotatingFileHandler`** with `maxBytes` and `backupCount` set. Add the `logs/` directory to `.gitignore` — logs should never be committed to version control.

```bash
# .gitignore
logs/
*.log
```

In [ ]:

print("=== Sensitive Data in Logs ===")
print()
dangerous = """
# CATASTROPHICALLY DANGEROUS
app.logger.info(f"Login: email={email}, password={password}")
# If this log file leaks: every user's password is exposed.
# This has caused real company-ending security breaches.
"""
safe = """
# SAFE -- log only non-sensitive identifiers
app.logger.info(f"Successful login: user_id={user.id}")
app.logger.warning(f"Failed login: {email[:3]}***")
"""
print("DANGEROUS:", dangerous)
print("SAFE:", safe)
print("Never log: passwords, tokens, credit cards, SSNs, OAuth tokens")

print()
print("=== Log File Growth ===")
print()
import math
print(f"{'Scenario':<30} {'MB/day':>10} {'GB/year':>10}")
print("-" * 52)
for reqs_per_sec, kb_per_req in [(1, 1), (10, 1), (100, 1), (100, 5)]:
    mb_per_day = reqs_per_sec * 86400 * kb_per_req / 1024
    gb_per_year = mb_per_day * 365 / 1024
    label = f"{reqs_per_sec} req/s, {kb_per_req}KB/req"
    print(f"  {label:<28} {mb_per_day:>10.1f} {gb_per_year:>10.1f}")

solution = """
from logging.handlers import RotatingFileHandler
handler = RotatingFileHandler(
    "logs/app.log",
    maxBytes=10 * 1024 * 1024,  # 10 MB per file
    backupCount=10               # max 110 MB total -- always bounded
)
"""
print()
print("Solution: RotatingFileHandler")
print(solution)


## 🚀 Real-World Project Link

In our **Blog Application**:

```text
flask-blog/
├── app/
│   ├── errors/
│   │   ├── __init__.py         <- Blueprint definition
│   │   ├── handlers.py         <- @errorhandler + db.session.rollback()
│   │   └── templates/errors/
│   │       ├── 404.html        <- styled "Not Found" page
│   │       ├── 403.html        <- styled "Forbidden" page
│   │       └── 500.html        <- styled "Server Error" with apology
│   └── __init__.py             <- configure_logging() in create_app()
├── logs/
│   ├── .gitignore              <- IMPORTANT: exclude logs from git!
│   └── app.log                 <- RotatingFileHandler target
└── config.py                   <- per-environment DEBUG, TESTING, log level
```

**What gets logged:** every login/logout (INFO), post CRUD (INFO), failed auth + IP (WARNING), all 500 errors with traceback (ERROR). Sentry captures production exceptions with user context. Error pages match the blog brand.

> ❓ **Why split code into Blueprints?** A single file with hundreds of routes becomes unreadable. Blueprints group related routes (e.g., all auth routes) into pluggable mini-applications that you register on the main app — like separate chapters in a book.

## 📋 Chapter Summary & Cheat Sheet

### Custom Error Handlers

```python
@app.errorhandler(404)
def not_found(error):
    return render_template('errors/404.html'), 404

@app.errorhandler(500)
def server_error(error):
    db.session.rollback()   # ALWAYS rollback on 500
    return render_template('errors/500.html'), 500
```
### Custom Exceptions

```python
from werkzeug.exceptions import HTTPException
class PostNotFound(HTTPException):
    code = 404
    description = "Post does not exist."
```
### Triggering Errors

```python
abort(404)                          # immediate inline
raise PostNotFound()                # named, semantic
post = Post.query.get_or_404(id)    # SQLAlchemy shortcut
```
### Production Logging

The important beginner shift here is moving from “it works on my machine” to “it behaves predictably in a real environment.” Each step matters because it reduces surprises when the software runs elsewhere.

```python
from logging.handlers import RotatingFileHandler
handler = RotatingFileHandler('logs/app.log', maxBytes=10_000_000, backupCount=10)
handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s: %(message)s'))
app.logger.addHandler(handler)
app.logger.setLevel(logging.INFO)
```
### Log Levels

```python
app.logger.debug("dev detail")                    # filtered in production
app.logger.info("user 42 logged in")              # normal events
app.logger.warning("disk 90% full")               # investigate soon
app.logger.error("payment failed", exc_info=True) # serious + traceback
app.logger.critical("db unreachable")             # immediate action
```
### Security Rules

| Rule | Why |
|------|-----|
| Never `DEBUG=True` in production | Remote code execution |
| Never log passwords/tokens | Log leaks expose all users |
| Always `db.session.rollback()` in 500 handler | Prevents cascading DB failures |
| Use `RotatingFileHandler` | Prevents disk-filling logs |
| Add Sentry in production | Real-time error alerts |

## 💪 Practice Prompts

**1. Custom Error Pages**
Create a Flask app with custom HTML handlers for 404, 403, and 500. The 404 page should include a search bar and home link. The 403 page should explain "forbidden" in plain English. The 500 page should include a contact email and apology. Test by visiting non-existent routes and calling `abort(403)`.

**2. Structured Logging**
Add production-ready logging to a Flask app. Configure `RotatingFileHandler` writing to `logs/app.log` (10 MB rotation, 10 backups). Add `logger.info()` on every successful POST, `logger.warning()` on failed logins, and `logger.error(exc_info=True)` in your 500 handler. Run the app, trigger events, then inspect the log file format.

**3. Custom Exception Classes**
Define 3+ custom `HTTPException` subclasses for a domain of your choice (e.g., `ArticleNotFound`, `CommentLocked`, `ProfilePrivate`). Register `app.errorhandler` for each. Replace all bare `abort()` calls in your routes with these named exceptions and notice how the code becomes more readable.

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='14.%20file_uploads_and_static_assets.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#8592; Chapter 14: File Uploads</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#128218; Table of Contents</a>
  <a href='16.%20testing_flask_applications.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 16: Testing &#8594;</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='14. file_uploads_and_static_assets.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='16. testing_flask_applications.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>